# Processing the Dataset

The dataset used is a compilation of 2126 BBC news articles ranging in different topics (politics, business, entertainment, sport and tech) from here: https://www.kaggle.com/datasets/alfathterry/bbc-full-text-document-classification, retrieved on November 1, 2025.

### Corpus Tokenization and N-gram generation

In [ ]:
import pandas as pd

# Function to load and preprocess the BBC News Summary dataset
def preprocessing_corpus(sourcepath, mt="word", N=2, nr_test_articles=2):

	df = pd.read_csv(sourcepath)
	df.columns = ["Article", "Category"]

	# Sort the articles in the dataframe by their category
	df = df.sort_values(by=['Category']).reset_index(drop=True)

	# Convert all text to lowercase
	#df['Article'] = df['Article'].str.lower()

	# Replace unicode literal for pound sign
	df = df.replace("xc2xa3", "£", regex=True)

	if mt == "word":
		# Add String "<s> " at the beginning of each article
		df['Article'] = "<s> " + df['Article'] + " </s>"

		# For all articles start cleaning by replacing single, double or triple full stops characters with "</s> <s>"
		df = df.replace(r"\.", " .", regex=True)
		df = df.replace(r"\!", " !", regex=True)
		df = df.replace(r"\?", " ?", regex=True)

		# Remove all appearances of the characters ( ) [ ] { } , " : ; from the texts
		chars_to_remove = [r"\(", r"\)", r"\[", r"\]", r"\{", r"\}", r"\,", r'"', r"\:", r"\;"]
		for char in chars_to_remove:
			df = df.replace(char, "", regex=True)

		# Replace multiple spaces (1, 2 or 3) with a single space	
		df = df.replace(r"\ {2,3}", " ", regex=True)

		# Split each text into a list of words
		words = lambda text: text.split(" ")
		df['Article'] = df['Article'].apply(words)

	else:
		# Replace multiple spaces (1, 2 or 3) with a single space	
		df = df.replace(r"\ {2,3}", " ", regex=True)
		
		# Split each text into a list of characters
		chars = lambda text: list(text)
		df['Article'] = df['Article'].apply(chars)

	ngram_list_train = []
	category_list_train = []
	ngram_list_test = []
	category_list_test = []

	# Generate n-grams from all but the last two articles from each category and store them in a new DataFrame	
	for index, row in df.iterrows():
		category = row['Category']
		article = row['Article']

		# Check the category of the next article
		if index < len(df) - nr_test_articles:
			next_category = df.at[index + nr_test_articles, 'Category']
		
		# If the category of the next article is different, process this article as test article
		if category != next_category or index >= len(df) - nr_test_articles:
			# Generate all n-grams for this article at once
			for i in range(len(article) - N + 1):
				ngram = article[i:i+N]
				ngram_list_test.append(ngram)
				category_list_test.append(category)

		else:
			# Generate all n-grams for this article at once
			for i in range(len(article) - N + 1):
				ngram = article[i:i+N]
				ngram_list_train.append(ngram)
				category_list_train.append(category)

		print(f"Processing corpus for {mt}-based {N}-gram: {index/(len(df)-1)*100:.2f}% processed", end="\r")
	
	# Create and return DataFrames for training and test n-grams
	ngrams_training = pd.DataFrame({'N-gram': ngram_list_train, 'Category': category_list_train})
	ngrams_test = pd.DataFrame({'N-gram': ngram_list_test, 'Category': category_list_test})
    
	return ngrams_training, ngrams_test

In [ ]:
# Preprocess text corpus and store ngrams to a new CSV file
N = {1, 2, 3, 4, 5}
modeltype = {"word", "character"}
nr_test_articles = 2
sourcepath = "./../Dataset/bbc_data.csv"
targetfolder = "./../Dataset_NGrams/"

for n in N:
    for mt in modeltype:
        ngrams_training, ngrams_test = preprocessing_corpus(sourcepath, mt, n, nr_test_articles)
        ngrams_training.to_csv(f"{targetfolder}bbc_{mt}_{n}grams_training.csv", index=False)
        ngrams_test.to_csv(f"{targetfolder}bbc_{mt}_{n}grams_test.csv", index=False)

### Frequencies

- Frequency dictionary from the N-gram CSV files generated earlier
- The goal is to count how often each possible continuation occurs after a given prefix.
	- These dictionaries are created respectively for the different categories of articles.

In [1]:
from collections import defaultdict
from ast import literal_eval  # to convert string list ↦ python list

def build_ngram_frequency_dict(csv_path, category):
	freq_dict = defaultdict(lambda: defaultdict(int))
	df = pd.read_csv(csv_path)

	# Reduce Dataframe to rows with specified categories
	df = df[df["Category"].isin(category)].reset_index(drop=True)

	# Process each N-gram row
	for ngram_str in df["N-gram"]:
		# literal_eval converts "['I','am','very']" → ['I','am','very']
		ngram = literal_eval(ngram_str)

		prefix = tuple(ngram[:-1])   	# all except last element
		next_token = ngram[-1]       	# last element

		# Count the next token
		freq_dict[prefix][next_token] += 1

		# Count total
		freq_dict[prefix]["<TOTAL>"] += 1
		
	return freq_dict

In [8]:
import os
import json
import pandas as pd

N = {1, 2, 3, 4, 5}
modeltype = {"word", "character"}
category = "all"				# business, entertainment, politics, sport, tech, all
sourcefolder = "./../Dataset_NGrams/"
targetfolder = "./../Frequency_NGrams/"

if category == "all":
	categories = {"business", "entertainment", "politics", "sport", "tech"}
else:
	categories = {category}

for n in N:
	for mt in modeltype:
		csv_file = f"{sourcefolder}bbc_{mt}_{n}grams_training.csv"
		
		# Skip non-existing files
		if not os.path.exists(csv_file):
			print(f"Skipping (file not found): {csv_file}")
			continue

		# Build frequency dictionary
		freq = build_ngram_frequency_dict(csv_file, categories)

		# Convert tuple keys → strings for JSON
		json_freq = {str(k): v for k, v in freq.items()}

		# Output filename
		output_path = f"{targetfolder}{mt}_{n}grams_{category}_frequency.json"

		# Save
		with open(output_path, "w", encoding="utf-8") as f:
			json.dump(json_freq, f, indent=4, ensure_ascii=False)

		print(f"Saved: {output_path}")

Saved: ./../Frequency_NGrams/character_1grams_all_frequency.json
Saved: ./../Frequency_NGrams/word_1grams_all_frequency.json
Saved: ./../Frequency_NGrams/character_2grams_all_frequency.json
Saved: ./../Frequency_NGrams/word_2grams_all_frequency.json
Saved: ./../Frequency_NGrams/character_3grams_all_frequency.json
Saved: ./../Frequency_NGrams/word_3grams_all_frequency.json
Saved: ./../Frequency_NGrams/character_4grams_all_frequency.json
Saved: ./../Frequency_NGrams/word_4grams_all_frequency.json
Saved: ./../Frequency_NGrams/character_5grams_all_frequency.json
Saved: ./../Frequency_NGrams/word_5grams_all_frequency.json


### Unigram vs. N-gram 

#### Unigram:
- A unigram has no prefix — it consists of a single token
- For unigrams, the frequency dictionary counts how many times each token appears

#### N-gram:

For example, in a trigram (3-gram) model:

- Prefix = (“I”, “am”)
- Next token = “tired”
- If the sequence "I am tired" appears 5 times in the corpus, then: ("I", "am") → { "tired": 5 }

For all n-grams, the first N − 1 elements form a prefix

The frequency dictionary stores, for each prefix:

- all possible next tokens
- how many times each continuation occurs
- the total count of sequences beginning with that prefix

This allows us to later build full probability models for text generation or language modeling:

$$
P(\text{next token} \mid \text{prefix})
\;=\;
\frac{\text{count(prefix → next)}}{\text{TOTAL(prefix)}}
$$


### Probabilities

In [10]:
from collections import defaultdict
from ast import literal_eval  # to convert string list ↦ python list

def build_ngram_probability_dict(freq_json):
	# Convert string keys back to tuple
	prob_dict = {}
	for prefix_str, next_tokens in freq_json.items():
		prefix = literal_eval(prefix_str)
		total_count = next_tokens.get("<TOTAL>", 0)
		
		# Calculate probabilities
		prob_dict[prefix] = {}
		for token, count in next_tokens.items():
			if token != "<TOTAL>":
				prob_dict[prefix][token] = count / total_count if total_count > 0 else 0.0

	return prob_dict

In [13]:
import os
import json

N = {1, 2, 3, 4, 5}
modeltype = {"word", "character"}
categories = {"business", "entertainment", "politics", "sport", "tech", "all"}
sourcefolder = "./../Frequency_NGrams/"
targetfolder = "./../Probability_NGrams/"

for category in categories:
	for n in N:
		for mt in modeltype:
			# Read in JSON frequency file and catch non-existing files
			json_path = f"{sourcefolder}{mt}_{n}grams_{category}_frequency.json"
			if not os.path.exists(json_path):
				print(f"Skipping (file not found): {json_path}")
				continue
			
			with open(json_path, "r", encoding="utf-8") as f:
				freq_json = json.load(f)

			# Build probability dictionary
			prob = build_ngram_probability_dict(freq_json)

			# Convert tuple keys → strings for JSON
			json_prob = {str(k): v for k, v in prob.items()}

			# Output filename
			output_path = f"{targetfolder}{mt}_{n}grams_{category}_probability.json"

			# Save
			with open(output_path, "w", encoding="utf-8") as f:
				json.dump(json_prob, f, indent=4, ensure_ascii=False)

			print(f"Saved: {output_path}")

Saved: ./../Probability_NGrams/character_1grams_business_probability.json
Saved: ./../Probability_NGrams/word_1grams_business_probability.json
Saved: ./../Probability_NGrams/character_2grams_business_probability.json
Saved: ./../Probability_NGrams/word_2grams_business_probability.json
Saved: ./../Probability_NGrams/character_3grams_business_probability.json
Saved: ./../Probability_NGrams/word_3grams_business_probability.json
Saved: ./../Probability_NGrams/character_4grams_business_probability.json
Saved: ./../Probability_NGrams/word_4grams_business_probability.json
Saved: ./../Probability_NGrams/character_5grams_business_probability.json
Saved: ./../Probability_NGrams/word_5grams_business_probability.json
Saved: ./../Probability_NGrams/character_1grams_sport_probability.json
Saved: ./../Probability_NGrams/word_1grams_sport_probability.json
Saved: ./../Probability_NGrams/character_2grams_sport_probability.json
Saved: ./../Probability_NGrams/word_2grams_sport_probability.json
Saved: ./..